In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

In [5]:
# llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
llm = ChatGroq(model="openai/gpt-oss-120b")

In [6]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [7]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [8]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [9]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [10]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it needed the dough, wanted a slice of the action, and heard the workplace had great “toppings” for career growth! 🍕😄',
 'explanation': '**Why the joke works**\n\n| Part of the punch‑line | What it literally means (pizza‑style) | What it means in everyday language | Why it’s funny |\n|------------------------|----------------------------------------|------------------------------------|----------------|\n| **“It needed the dough”** | Pizza dough is the flour‑water mixture that makes the crust. | “Dough” is slang for money. | The joke swaps a literal ingredient for a slang term, making us picture a pizza worrying about its paycheck. |\n| **“Wanted a slice of the action”** | A pizza is cut into slices; a “slice” is a piece of the whole pie. | “A slice of the action” is an idiom meaning “a share of what’s going on, a chance to benefit.” | The double meaning of “slice” (pizza piece vs. share) creates a pun that ties

In [11]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it needed the dough, wanted a slice of the action, and heard the workplace had great “toppings” for career growth! 🍕😄', 'explanation': '**Why the joke works**\n\n| Part of the punch‑line | What it literally means (pizza‑style) | What it means in everyday language | Why it’s funny |\n|------------------------|----------------------------------------|------------------------------------|----------------|\n| **“It needed the dough”** | Pizza dough is the flour‑water mixture that makes the crust. | “Dough” is slang for money. | The joke swaps a literal ingredient for a slang term, making us picture a pizza worrying about its paycheck. |\n| **“Wanted a slice of the action”** | A pizza is cut into slices; a “slice” is a piece of the whole pie. | “A slice of the action” is an idiom meaning “a share of what’s going on, a chance to benefit.” | The double meaning of “slice” (pizza piece vs. share) crea

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it needed the dough, wanted a slice of the action, and heard the workplace had great “toppings” for career growth! 🍕😄', 'explanation': '**Why the joke works**\n\n| Part of the punch‑line | What it literally means (pizza‑style) | What it means in everyday language | Why it’s funny |\n|------------------------|----------------------------------------|------------------------------------|----------------|\n| **“It needed the dough”** | Pizza dough is the flour‑water mixture that makes the crust. | “Dough” is slang for money. | The joke swaps a literal ingredient for a slang term, making us picture a pizza worrying about its paycheck. |\n| **“Wanted a slice of the action”** | A pizza is cut into slices; a “slice” is a piece of the whole pie. | “A slice of the action” is an idiom meaning “a share of what’s going on, a chance to benefit.” | The double meaning of “slice” (pizza piece vs. share) cre

In [13]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': "Why did the spaghetti go to therapy?  \n\nBecause it couldn't stop **twisting** itself into knots and felt **sauced** out of control! 🍝😆",
 'explanation': "**The joke**\n\n> *Why did the spaghetti go to therapy?*  \n> *Because it couldn't stop **twisting** itself into knots and felt **sauced** out of control!* 🍝😆  \n\n---\n\n## 1. The literal side – what spaghetti actually does  \n\n1. **Twisting into knots** – When you eat long pasta like spaghetti, you often twirl it around a fork. The strands curl, loop, and sometimes even tangle into little knots.  \n2. **Sauced** – Spaghetti is almost always served with sauce (tomato, pesto, alfredo, etc.). “Sauced” is a common slang term for being drunk or “under the influence,” but here it’s a literal reference to the pasta being covered in sauce.\n\n---\n\n## 2. The figurative side – the mental‑health metaphor  \n\n- **Therapy** is where people (or, in a joke, anything) go when they feel they can’t manage a recurrin

In [14]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the spaghetti go to therapy?  \n\nBecause it couldn't stop **twisting** itself into knots and felt **sauced** out of control! 🍝😆", 'explanation': "**The joke**\n\n> *Why did the spaghetti go to therapy?*  \n> *Because it couldn't stop **twisting** itself into knots and felt **sauced** out of control!* 🍝😆  \n\n---\n\n## 1. The literal side – what spaghetti actually does  \n\n1. **Twisting into knots** – When you eat long pasta like spaghetti, you often twirl it around a fork. The strands curl, loop, and sometimes even tangle into little knots.  \n2. **Sauced** – Spaghetti is almost always served with sauce (tomato, pesto, alfredo, etc.). “Sauced” is a common slang term for being drunk or “under the influence,” but here it’s a literal reference to the pasta being covered in sauce.\n\n---\n\n## 2. The figurative side – the mental‑health metaphor  \n\n- **Therapy** is where people (or, in a joke, anything) go when they feel they can’

In [15]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the spaghetti go to therapy?  \n\nBecause it couldn't stop **twisting** itself into knots and felt **sauced** out of control! 🍝😆", 'explanation': "**The joke**\n\n> *Why did the spaghetti go to therapy?*  \n> *Because it couldn't stop **twisting** itself into knots and felt **sauced** out of control!* 🍝😆  \n\n---\n\n## 1. The literal side – what spaghetti actually does  \n\n1. **Twisting into knots** – When you eat long pasta like spaghetti, you often twirl it around a fork. The strands curl, loop, and sometimes even tangle into little knots.  \n2. **Sauced** – Spaghetti is almost always served with sauce (tomato, pesto, alfredo, etc.). “Sauced” is a common slang term for being drunk or “under the influence,” but here it’s a literal reference to the pasta being covered in sauce.\n\n---\n\n## 2. The figurative side – the mental‑health metaphor  \n\n- **Therapy** is where people (or, in a joke, anything) go when they feel they can

**Time Travel**

In [19]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1c9671-94ba-688e-8000-65777706e85a"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1c9671-94ba-688e-8000-65777706e85a'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-16T13:40:06.839904+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c9671-94a7-6d8a-bfff-5154e4b34ea1'}}, tasks=(PregelTask(id='72a9092c-99ff-9bb3-c6e0-b22d4a2d79f5', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza apply for a job?\n\nBecause it needed the dough, wanted a slice of the action, and heard the workplace had great “toppings” for career growth! 🍕😄'}),), interrupts=())

In [20]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1c9671-94ba-688e-8000-65777706e85a"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company was looking for someone with *dough*‑mestic skills and a *crust*‑worthy work ethic! 🍕😄',
 'explanation': '**Joke:**  \n*Why did the pizza apply for a job?*  \n*Because it heard the company was looking for someone with **dough‑mestic** skills and a **crust‑worthy** work ethic! 🍕😄*\n\n---\n\n### Why it’s funny – a step‑by‑step breakdown\n\n| Part of the punchline | What it literally means | The wordplay (pun) | Why it makes you smile |\n|-----------------------|--------------------------|--------------------|------------------------|\n| **dough‑mestic** | “Domestic” normally means “relating to the home” or “household.” | Replaces the “do‑” in *domestic* with **dough**, the soft, buttery base of a pizza. | It turns a common job‑ad phrase (“domestic skills”) into a pizza‑themed gag. The mental image of a pizza having “dough‑related” abilities is absurd and cute. |\n| **crust‑worthy** | “Worth‑

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company was looking for someone with *dough*‑mestic skills and a *crust*‑worthy work ethic! 🍕😄', 'explanation': '**Joke:**  \n*Why did the pizza apply for a job?*  \n*Because it heard the company was looking for someone with **dough‑mestic** skills and a **crust‑worthy** work ethic! 🍕😄*\n\n---\n\n### Why it’s funny – a step‑by‑step breakdown\n\n| Part of the punchline | What it literally means | The wordplay (pun) | Why it makes you smile |\n|-----------------------|--------------------------|--------------------|------------------------|\n| **dough‑mestic** | “Domestic” normally means “relating to the home” or “household.” | Replaces the “do‑” in *domestic* with **dough**, the soft, buttery base of a pizza. | It turns a common job‑ad phrase (“domestic skills”) into a pizza‑themed gag. The mental image of a pizza having “dough‑related” abilities is absurd and cute. |\n| **crus

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company was looking for someone with *dough*‑mestic skills and a *crust*‑worthy work ethic! 🍕😄', 'explanation': '**Joke:**  \n*Why did the pizza apply for a job?*  \n*Because it heard the company was looking for someone with **dough‑mestic** skills and a **crust‑worthy** work ethic! 🍕😄*\n\n---\n\n### Why it’s funny – a step‑by‑step breakdown\n\n| Part of the punchline | What it literally means | The wordplay (pun) | Why it makes you smile |\n|-----------------------|--------------------------|--------------------|------------------------|\n| **dough‑mestic** | “Domestic” normally means “relating to the home” or “household.” | Replaces the “do‑” in *domestic* with **dough**, the soft, buttery base of a pizza. | It turns a common job‑ad phrase (“domestic skills”) into a pizza‑themed gag. The mental image of a pizza having “dough‑related” abilities is absurd and cute. |\n| **crus

**Fault Tolerance**

In [24]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [25]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [26]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [27]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))